# EmoWOZ Download + FinDisputeEval Emotion EDA

This notebook is intended for VS Code with a Colab runtime. It mounts Google Drive, downloads EmoWOZ source JSON files from Zenodo, reconstructs train/validation/test splits, and runs EDA focused on FinDisputeEval emotion and escalation handling.

The Hugging Face dataset viewer for `hhu-dsml/emowoz` is disabled because the dataset repo requires arbitrary Python code execution. This notebook avoids that issue by downloading the JSON files used by the official dataset script.

## Why FinDisputeEval Should Look at Dialogue Emotion

FinDisputeEval is not using EmoWOZ as a banking-dispute corpus. It uses EmoWOZ to model emotion-aware task-oriented dialogue behavior.

Consumer dispute intake often includes frustration, fear, apology/correction, anger, and pressure to escalate. Emotion should not decide the legal outcome, but it should affect response style and control flow:

- Dissatisfied users may need repair, acknowledgement, and clearer explanation.
- Fearful users may need simpler questions and reassurance without unsupported promises.
- Abusive language may require de-escalation and safe handoff behavior.
- Apologetic/corrective language often signals slot repair, such as changing debit to credit or correcting the amount/date.
- Satisfied users can help evaluate successful closure patterns.

For FinDisputeEval, emotion labels are best used for escalation, tone calibration, repair strategy, and human-handoff policy. They should not override Reg E / Reg Z routing or factual validation.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%pip -q install -U pandas pyarrow matplotlib

In [ ]:
from __future__ import annotations

import json
import os
import re
import urllib.request
from datetime import datetime, timezone
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "FinDisputeEval"
RUN_ID = globals().get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RAW_DIR = PROJECT_DIR / "dataset" / "external" / "emowoz" / "raw"
PROCESSED_DIR = PROJECT_DIR / "dataset" / "interim" / "emowoz" / "processed_findispute"
EDA_DIR = PROJECT_DIR / "outputs" / "data_pipeline" / "emowoz_emotion_eda" / "eda_v01" / f"run_{RUN_ID}_colab"
OUTPUT_DIR = EDA_DIR

for directory in [RAW_DIR, PROCESSED_DIR, EDA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ZENODO_FILES = {
    "emowoz-multiwoz.json": "https://zenodo.org/record/6506504/files/emowoz-multiwoz.json",
    "emowoz-dialmage.json": "https://zenodo.org/record/6506504/files/emowoz-dialmage.json",
    "data-split.json": "https://zenodo.org/record/6506504/files/data-split.json",
}

EMOTION_LABELS = {
    -1: "system_unlabeled",
    0: "neutral",
    1: "fearful_or_sad",
    2: "dissatisfied",
    3: "apologetic",
    4: "abusive",
    5: "excited",
    6: "satisfied",
}

SENTIMENT_GROUPS = {
    -1: "system_unlabeled",
    0: "neutral",
    1: "negative_fact_or_fear",
    2: "negative_system_dissatisfaction",
    3: "apology_or_correction",
    4: "abusive_escalation",
    5: "positive_excitement",
    6: "positive_satisfaction",
}

RANDOM_STATE = 42

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
def download_if_missing(url: str, target_path: Path) -> None:
    if target_path.exists() and target_path.stat().st_size > 0:
        print(f"Already exists: {target_path.name} ({target_path.stat().st_size:,} bytes)")
        return
    print(f"Downloading {target_path.name} ...")
    urllib.request.urlretrieve(url, target_path)
    print(f"Saved: {target_path} ({target_path.stat().st_size:,} bytes)")


for filename, url in ZENODO_FILES.items():
    download_if_missing(url, RAW_DIR / filename)

manifest = {
    "huggingface_dataset": "https://huggingface.co/datasets/hhu-dsml/emowoz",
    "zenodo_record": "https://zenodo.org/record/6506504",
    "raw_dir": str(RAW_DIR),
    "processed_dir": str(PROCESSED_DIR),
    "eda_dir": str(EDA_DIR),
    "files": {filename: {"url": url, "path": str(RAW_DIR / filename)} for filename, url in ZENODO_FILES.items()},
}

(OUTPUT_DIR / "download_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest saved to: {OUTPUT_DIR / 'download_manifest.json'}")

## Reconstruct Splits and Normalize Turns

In [ ]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def safe_text(value) -> str:
    if value is None:
        return ""
    return str(value)


def word_count(text: str) -> int:
    text = re.sub(r"\s+", " ", safe_text(text)).strip()
    return len(text.split()) if text else 0


def extract_user_emotion(turn: dict, turn_index: int) -> int:
    if turn_index % 2 == 1:
        return -1
    try:
        return int(turn.get("emotion", [None, None, None, {"emotion": 0}])[3]["emotion"])
    except Exception:
        return 0


multiwoz_dialogues = load_json(RAW_DIR / "emowoz-multiwoz.json")
dialmage_dialogues = load_json(RAW_DIR / "emowoz-dialmage.json")
data_split = load_json(RAW_DIR / "data-split.json")
dialogues = {**multiwoz_dialogues, **dialmage_dialogues}

split_map = {"train": "train", "validation": "dev", "test": "test"}
dialogue_rows = []
turn_rows = []

for split_name, source_split_name in split_map.items():
    source_by_id = {}
    for source_subset in ["multiwoz", "dialmage"]:
        for dialogue_id in data_split[source_split_name][source_subset]:
            source_by_id[dialogue_id] = source_subset

    for dialogue_id, source_subset in source_by_id.items():
        dialogue = dialogues[dialogue_id]
        log = dialogue.get("log", [])
        dialogue_uid = f"{split_name}-{dialogue_id}"
        user_emotions = []
        full_text_parts = []

        for turn_index, turn in enumerate(log):
            speaker = "user" if turn_index % 2 == 0 else "system"
            text = safe_text(turn.get("text", ""))
            emotion_id = extract_user_emotion(turn, turn_index)
            emotion_label = EMOTION_LABELS.get(emotion_id, f"unknown_{emotion_id}")
            sentiment_group = SENTIMENT_GROUPS.get(emotion_id, "unknown")
            full_text_parts.append(text)

            if speaker == "user":
                user_emotions.append(emotion_id)

            turn_rows.append({
                "dialogue_uid": dialogue_uid,
                "dialogue_id": dialogue_id,
                "split": split_name,
                "source_subset": source_subset,
                "turn_index": turn_index,
                "speaker": speaker,
                "text": text,
                "word_count": word_count(text),
                "emotion_id": emotion_id,
                "emotion_label": emotion_label,
                "sentiment_group": sentiment_group,
                "is_user_turn": speaker == "user",
                "is_negative_emotion": emotion_id in {1, 2, 4},
                "is_positive_emotion": emotion_id in {5, 6},
            })

        emotion_counts = Counter(user_emotions)
        first_user_emotion = user_emotions[0] if user_emotions else None
        last_user_emotion = user_emotions[-1] if user_emotions else None
        has_negative = any(emotion in {1, 2, 4} for emotion in user_emotions)
        has_positive = any(emotion in {5, 6} for emotion in user_emotions)
        negative_to_positive = has_negative and last_user_emotion in {5, 6}

        dialogue_rows.append({
            "dialogue_uid": dialogue_uid,
            "dialogue_id": dialogue_id,
            "split": split_name,
            "source_subset": source_subset,
            "turn_count": len(log),
            "user_turn_count": len(user_emotions),
            "system_turn_count": len(log) - len(user_emotions),
            "dialogue_word_count": word_count(" ".join(full_text_parts)),
            "first_user_emotion_id": first_user_emotion,
            "first_user_emotion_label": EMOTION_LABELS.get(first_user_emotion, "none"),
            "last_user_emotion_id": last_user_emotion,
            "last_user_emotion_label": EMOTION_LABELS.get(last_user_emotion, "none"),
            "has_fearful_or_sad": emotion_counts.get(1, 0) > 0,
            "has_dissatisfied": emotion_counts.get(2, 0) > 0,
            "has_apologetic": emotion_counts.get(3, 0) > 0,
            "has_abusive": emotion_counts.get(4, 0) > 0,
            "has_excited": emotion_counts.get(5, 0) > 0,
            "has_satisfied": emotion_counts.get(6, 0) > 0,
            "has_negative": has_negative,
            "has_positive": has_positive,
            "negative_to_positive": negative_to_positive,
            "emotion_sequence": " > ".join(EMOTION_LABELS.get(e, str(e)) for e in user_emotions),
            "full_text": " ".join(full_text_parts),
        })

dialogue_df = pd.DataFrame(dialogue_rows)
turn_df = pd.DataFrame(turn_rows)

dialogue_df.to_parquet(PROCESSED_DIR / "emowoz_dialogues.parquet", index=False)
turn_df.to_parquet(PROCESSED_DIR / "emowoz_turns.parquet", index=False)
dialogue_df.to_csv(PROCESSED_DIR / "emowoz_dialogues.csv", index=False)
turn_df.to_csv(PROCESSED_DIR / "emowoz_turns.csv", index=False)

print(f"Dialogue rows: {len(dialogue_df):,}")
print(f"Turn rows: {len(turn_df):,}")
print(f"User emotion annotations: {(turn_df['emotion_id'] != -1).sum():,}")
display(dialogue_df.head())
display(turn_df.head(12))

## Dataset Shape and Emotion Distribution

In [ ]:
split_summary = (
    dialogue_df.groupby(["split", "source_subset"], dropna=False)
    .agg(
        dialogues=("dialogue_uid", "count"),
        avg_turns=("turn_count", "mean"),
        p50_turns=("turn_count", "median"),
        p95_turns=("turn_count", lambda s: s.quantile(0.95)),
        avg_user_turns=("user_turn_count", "mean"),
        avg_words=("dialogue_word_count", "mean"),
        negative_dialogues=("has_negative", "sum"),
        positive_dialogues=("has_positive", "sum"),
        negative_to_positive=("negative_to_positive", "sum"),
    )
    .reset_index()
)
split_summary["negative_dialogue_rate"] = split_summary["negative_dialogues"] / split_summary["dialogues"]
split_summary["positive_dialogue_rate"] = split_summary["positive_dialogues"] / split_summary["dialogues"]

user_turns = turn_df[turn_df["is_user_turn"]].copy()
emotion_distribution = (
    user_turns.groupby(["emotion_id", "emotion_label"], dropna=False)
    .agg(turns=("dialogue_uid", "count"), dialogues=("dialogue_uid", "nunique"), avg_words=("word_count", "mean"))
    .reset_index()
    .sort_values("turns", ascending=False)
)
emotion_distribution["turn_rate"] = emotion_distribution["turns"] / len(user_turns)

emotion_by_split = (
    user_turns.groupby(["split", "source_subset", "emotion_label"], dropna=False)
    .size()
    .reset_index(name="turns")
    .sort_values(["split", "source_subset", "turns"], ascending=[True, True, False])
)

for filename, df in {
    "emowoz_split_summary.csv": split_summary,
    "emowoz_emotion_distribution.csv": emotion_distribution,
    "emowoz_emotion_by_split.csv": emotion_by_split,
}.items():
    df.to_csv(EDA_DIR / filename, index=False)

display(split_summary)
display(emotion_distribution)
display(emotion_by_split.head(50))

## FinDisputeEval-Oriented Emotion Roles

In [ ]:
KEYWORD_GROUPS = {
    "confusion_repair": [r"confus", r"clarify", r"sorry", r"apolog", r"mistake", r"wrong", r"change", r"instead", r"actually"],
    "frustration_escalation": [r"angry", r"upset", r"annoy", r"bad", r"terrible", r"ridiculous", r"complain", r"manager", r"supervisor"],
    "urgency_risk": [r"urgent", r"asap", r"immediately", r"right now", r"today", r"late", r"miss", r"problem"],
    "money_payment": [r"money", r"pay", r"paid", r"payment", r"price", r"cost", r"refund", r"charge", r"bill"],
    "account_booking": [r"account", r"book", r"booking", r"reference", r"reservation", r"cancel"],
}

for group_name, patterns in KEYWORD_GROUPS.items():
    regex = "|".join(patterns)
    dialogue_df[f"kw_{group_name}"] = dialogue_df["full_text"].str.lower().str.contains(regex, regex=True, na=False)
    turn_df[f"kw_{group_name}"] = turn_df["text"].str.lower().str.contains(regex, regex=True, na=False)

keyword_cols = [f"kw_{name}" for name in KEYWORD_GROUPS]
dialogue_df["keyword_hit_count"] = dialogue_df[keyword_cols].sum(axis=1)
turn_df["keyword_hit_count"] = turn_df[keyword_cols].sum(axis=1)


def assign_dialogue_role(row) -> str:
    if row["has_abusive"]:
        return "abusive_escalation_stress"
    if row["negative_to_positive"]:
        return "de_escalation_recovery_candidate"
    if row["has_dissatisfied"]:
        return "dissatisfaction_repair_candidate"
    if row["has_fearful_or_sad"]:
        return "fear_or_uncertainty_sensitive_support"
    if row["has_apologetic"]:
        return "apology_or_slot_correction_candidate"
    if row["has_satisfied"]:
        return "successful_completion_positive_tone"
    return "neutral_task_dialogue_baseline"


dialogue_df["findispute_emotion_role"] = dialogue_df.apply(assign_dialogue_role, axis=1)

role_summary = (
    dialogue_df.groupby(["findispute_emotion_role"], dropna=False)
    .agg(
        dialogues=("dialogue_uid", "count"),
        avg_turns=("turn_count", "mean"),
        avg_words=("dialogue_word_count", "mean"),
        negative_dialogue_rate=("has_negative", "mean"),
        positive_dialogue_rate=("has_positive", "mean"),
        keyword_hit_rate=("keyword_hit_count", lambda s: (s > 0).mean()),
    )
    .reset_index()
    .sort_values("dialogues", ascending=False)
)

role_by_split = (
    dialogue_df.groupby(["split", "source_subset", "findispute_emotion_role"], dropna=False)
    .size()
    .reset_index(name="dialogues")
    .sort_values(["split", "source_subset", "dialogues"], ascending=[True, True, False])
)

keyword_summary = (
    dialogue_df.melt(
        id_vars=["split", "source_subset", "findispute_emotion_role"],
        value_vars=keyword_cols,
        var_name="keyword_group",
        value_name="hit",
    )
    .query("hit")
    .groupby(["keyword_group", "source_subset"], dropna=False)
    .size()
    .reset_index(name="dialogues")
    .sort_values(["keyword_group", "dialogues"], ascending=[True, False])
)

dialogue_df.to_parquet(PROCESSED_DIR / "emowoz_dialogues_with_findispute_roles.parquet", index=False)
turn_df.to_parquet(PROCESSED_DIR / "emowoz_turns_with_keywords.parquet", index=False)
dialogue_df.to_csv(PROCESSED_DIR / "emowoz_dialogues_with_findispute_roles.csv", index=False)
turn_df.to_csv(PROCESSED_DIR / "emowoz_turns_with_keywords.csv", index=False)
role_summary.to_csv(EDA_DIR / "emowoz_findispute_emotion_role_summary.csv", index=False)
role_by_split.to_csv(EDA_DIR / "emowoz_findispute_emotion_role_by_split.csv", index=False)
keyword_summary.to_csv(EDA_DIR / "emowoz_findispute_keyword_summary.csv", index=False)

display(role_summary)
display(role_by_split.head(60))
display(keyword_summary)

## Emotion Context Windows

These rows are useful for training/evaluating response-style decisions: the previous system turn, the emotional user turn, and the next system turn.

In [ ]:
turn_df = turn_df.sort_values(["dialogue_uid", "turn_index"]).reset_index(drop=True)
turn_df["prev_speaker"] = turn_df.groupby("dialogue_uid")["speaker"].shift(1)
turn_df["prev_text"] = turn_df.groupby("dialogue_uid")["text"].shift(1)
turn_df["next_speaker"] = turn_df.groupby("dialogue_uid")["speaker"].shift(-1)
turn_df["next_text"] = turn_df.groupby("dialogue_uid")["text"].shift(-1)

emotion_context_df = turn_df[
    turn_df["is_user_turn"] & turn_df["emotion_id"].isin([1, 2, 3, 4, 5, 6])
].copy()

context_cols = [
    "dialogue_uid", "split", "source_subset", "turn_index", "emotion_id", "emotion_label", "sentiment_group",
    "prev_speaker", "prev_text", "text", "next_speaker", "next_text", "keyword_hit_count",
]
emotion_context_df[context_cols].to_csv(EDA_DIR / "emowoz_emotion_context_windows.csv", index=False)

context_summary = (
    emotion_context_df.groupby(["emotion_label"], dropna=False)
    .agg(rows=("dialogue_uid", "count"), avg_words=("word_count", "mean"), keyword_hit_rate=("keyword_hit_count", lambda s: (s > 0).mean()))
    .reset_index()
    .sort_values("rows", ascending=False)
)
context_summary.to_csv(EDA_DIR / "emowoz_emotion_context_summary.csv", index=False)

display(context_summary)
display(emotion_context_df[context_cols].head(30))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 11))

emotion_distribution.set_index("emotion_label")["turns"].sort_values().plot(
    kind="barh", ax=axes[0, 0], title="User emotion distribution"
)
axes[0, 0].set_xlabel("user turns")
axes[0, 0].set_ylabel("")

split_summary.groupby("split")["dialogues"].sum().plot(kind="bar", ax=axes[0, 1], title="Dialogues by split")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("dialogues")

role_summary.sort_values("dialogues").plot(
    kind="barh", x="findispute_emotion_role", y="dialogues", ax=axes[1, 0], legend=False, title="FinDispute emotion-role coverage"
)
axes[1, 0].set_xlabel("dialogues")
axes[1, 0].set_ylabel("")

keyword_plot = keyword_summary.groupby("keyword_group")["dialogues"].sum().sort_values()
keyword_plot.plot(kind="barh", ax=axes[1, 1], title="Keyword coverage by dialogue")
axes[1, 1].set_xlabel("dialogues")
axes[1, 1].set_ylabel("")

plt.tight_layout()
plot_path = EDA_DIR / "emowoz_findispute_eda_overview.png"
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
print(f"Saved plot to: {plot_path}")
plt.show()

## FinDispute Emotion Seed Samples

Samples are context windows, not legal labels. Use them to design response-style and escalation tests.

In [ ]:
sample_specs = [
    ("dissatisfied_repair", "dissatisfied", 70),
    ("abusive_escalation", "abusive", 40),
    ("fearful_sensitive_support", "fearful_or_sad", 40),
    ("apology_or_slot_correction", "apologetic", 40),
    ("positive_satisfaction_closure", "satisfied", 40),
    ("positive_excited", "excited", 25),
]

seed_parts = []
used_index = set()
for bucket_name, emotion_label, target_n in sample_specs:
    pool = emotion_context_df[
        emotion_context_df["emotion_label"].eq(emotion_label)
        & ~emotion_context_df.index.isin(used_index)
    ]
    if pool.empty:
        print(f"No rows for {bucket_name} / {emotion_label}")
        continue
    n = min(target_n, len(pool))
    sampled = pool.sample(n=n, random_state=RANDOM_STATE).copy()
    sampled["seed_bucket"] = bucket_name
    used_index.update(sampled.index.tolist())
    seed_parts.append(sampled)

if seed_parts:
    seed_df = pd.concat(seed_parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    seed_df = emotion_context_df.sample(n=min(100, len(emotion_context_df)), random_state=RANDOM_STATE).copy()
    seed_df["seed_bucket"] = "fallback_random"

seed_cols = [
    "seed_bucket", "dialogue_uid", "split", "source_subset", "turn_index", "emotion_label", "sentiment_group",
    "prev_text", "text", "next_text", "keyword_hit_count",
]
seed_path = EDA_DIR / "emowoz_findispute_emotion_seed_contexts.csv"
seed_df[seed_cols].to_csv(seed_path, index=False)

print(f"Saved seed contexts: {seed_path}")
print(f"Seed rows: {len(seed_df):,}")
display(seed_df[seed_cols].head(40))

## Expected Output Structure

```text
dataset/external/emowoz/raw/
  emowoz-multiwoz.json
  emowoz-dialmage.json
  data-split.json

dataset/interim/emowoz/processed_findispute/
  emowoz_dialogues.csv / .parquet
  emowoz_turns.csv / .parquet
  emowoz_dialogues_with_findispute_roles.csv / .parquet
  emowoz_turns_with_keywords.parquet

outputs/data_pipeline/emowoz_emotion_eda/eda_v01/<run_id>/
  emowoz_split_summary.csv
  emowoz_emotion_distribution.csv
  emowoz_emotion_by_split.csv
  emowoz_findispute_emotion_role_summary.csv
  emowoz_findispute_keyword_summary.csv
  emowoz_emotion_context_windows.csv
  emowoz_findispute_eda_overview.png
  emowoz_findispute_emotion_seed_contexts.csv
```
